# Power Plant Sensor Forecasting Benchmark

This notebook is the narrative layer for the public-safe benchmark package. The `.py` script is the reproducible engine. This notebook is the readable walkthrough: why the benchmark exists, what changed during audit, and what the charts actually say.

All sensor names are masked. The goal is to preserve the technical story without exposing the raw plant schema.


## Benchmark snapshot

- 50,388 rows
- 23 sensor columns
- 174 days of history
- 5-minute sampling cadence
- No missing values
- Selected target: continuous emissions-related stack sensor
- Original heuristic failure: sparse candidate signal with 98.5% zero values


## How this notebook complements the script

- `scripts/run_power_plant_forecasting.py` runs the full benchmark pipeline.
- `data/public-summary.json` stores the sanitized benchmark summary used below.
- `assets/` contains the masked figures for public review.

The notebook is intentionally structured like a technical memo rather than a scratchpad.


In [ ]:
from pathlib import Path
import json

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
summary = json.loads((REPO_ROOT / 'data' / 'public-summary.json').read_text())
summary['dataset']


## Dataset profile

| Property | Value |
| --- | ---: |
| Rows | 50,388 |
| Sensor columns | 23 |
| Time span | 174 days |
| Start | 2005-01-01 00:05 |
| End | 2005-06-25 00:00 |
| Missing values | 0 |

This is enough history to benchmark short-horizon forecasting seriously, but also enough room for bad target selection to create misleading conclusions.


## Audit finding: the original heuristic picked the wrong kind of target

The original MVP ranked candidate targets using autocorrelation, variance, and outlier rate. That sounds reasonable until you inspect the operational profile of the winner.

The highest-ranked candidate was a sparse auxiliary fuel signal. It had excellent autocorrelation and still produced a broken downstream task because it was almost always inactive. That is the first lesson of the benchmark: target selection for industrial telemetry has to include activity coverage, not just time-series smoothness.


![Collapsed label distribution for the heuristic-selected target](../assets/heuristic-target-label-collapse.png)


## Selected target profile

The benchmark was rerun on the strongest non-sparse candidate: a continuous emissions-related stack sensor with strong short-range memory.

| Metric | Value |
| --- | ---: |
| Mean | 1426.84 |
| Median | 1392.96 |
| Std. dev. | 385.53 |
| Lag-1 autocorrelation | 0.9894 |
| Lag-2 autocorrelation | 0.9703 |
| Outlier rate | 4.65% |

The purpose of the benchmark is not to find the flashiest score. It is to put the model on a target that is actually worth forecasting.


![Selected target overview](../assets/target-series-overview.png)


## Temporal behavior

The signal carries clear temporal structure, but the effect sizes are modest compared with the dominance of recent history. That matters because it explains why lag features end up overpowering calendar features in the model.


![Temporal patterns](../assets/temporal-patterns.png)


## Support-sensor context

Cross-sensor correlations are present but not overwhelming. This is typical in plant telemetry: correlated process variables help, but the immediate value history of the target still does most of the predictive work.


![Support sensor correlations](../assets/sensor-correlations.png)


## Data split protocol

The benchmark uses a temporal train/validation/test split rather than random shuffling.

| Split | Rows |
| --- | ---: |
| Train | 35,261 |
| Validation | 7,556 |
| Test | 7,557 |

That preserves the operational sequence and keeps leakage out of the evaluation.


![Temporal train/validation/test split](../assets/dataset-split.png)


## Forecast traces

Both Random Forest and XGBoost track the target well at 5-minute and 10-minute horizons. The important nuance is that visual agreement alone is not enough. The benchmark still has to clear the persistence baseline.


![Forecast versus actual](../assets/forecast-vs-actual.png)


## Regression benchmark

| Horizon | Baseline R² | Random Forest R² | XGBoost R² | Baseline RMSE | Random Forest RMSE | XGBoost RMSE |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 5 min | 0.9707 | 0.9335 | 0.9326 | 44.48 | 67.04 | 67.51 |
| 10 min | 0.9274 | 0.8889 | 0.8878 | 70.15 | 86.78 | 87.19 |

This is the core result. Forecasting is feasible, but the persistence baseline is already extremely strong at these horizons. That means the value of the benchmark is less about declaring a model winner and more about defining the right bar for future work.


![Performance comparison](../assets/performance-comparison.png)


## Feature importance

The model tells a straightforward story: the previous value dominates everything else.

| Top feature | Importance |
| --- | ---: |
| Target lag t-1 | 0.9622 |
| Target rolling max (15m) | 0.0031 |
| Target lag t-2 | 0.0028 |
| Support sensor A | 0.0025 |
| Support sensor E | 0.0022 |

When `Target lag t-1` carries more than 96% of the importance mass, that is a signal to benchmark longer horizons and regime-aware objectives before reaching for additional model complexity.


![Feature importance](../assets/feature-importance.png)


## Classification benchmark

The trend-classification branch is directionally useful but not production-ready.

| Horizon | Random Forest accuracy | XGBoost accuracy |
| --- | ---: | ---: |
| 5 min | 0.5555 | 0.5538 |
| 10 min | 0.4171 | 0.4231 |

The confusion matrices show a strong pull toward the `Steady` class, especially at the shorter horizon. This is useful diagnostically, but it is not strong enough to become the main control surface.


![Confusion matrices](../assets/confusion-matrices.png)


## Takeaway

The strongest outcome of this project is not a single model metric. It is the evaluation discipline:

1. audit the heuristic
2. reject sparse pseudo-targets
3. benchmark against persistence first
4. only then decide whether model complexity is earning its keep

That is the standard the `.py` pipeline enforces, and this notebook is the public-facing explanation of that standard.


## Rerun privately

If you have authorized access to the original CSV, rerun the benchmark with the script below.

```bash
python scripts/run_power_plant_forecasting.py   --data /path/to/private_dataset.csv   --timestamp-column "<private timestamp column>"   --target "<private target column>"   --public-safe   --output-dir ./artifacts
```

The script remains the source of truth. The notebook exists to make the benchmark legible.
